# Selecting which parameters to fit

For complex simulations, there may be a lot of parameters; [a simple argon simulation](../../../tutorials/Argon-a-to-z.ipynb) has two parameters, and later on in this guide we will see a water simulation with a total of 8 - this only gets bigger as the molecules and forces become more complex.

Thus, a user may only want to refine a subset of all their parameters. In MDMC this can be done in two ways:

### Fixing a Parameter

`Parameter` objects can be fixed.  This can either be set when they are initialised (created) or changed for an existing `Parameter`.  By default, an initialised `Parameter` is **not** fixed.

In [ ]:
from MDMC.MD.parameters import Parameter
charge = Parameter(value=0.5, name='charge', fixed=True, unit='e')

Attempting to change a fixed parameter produces a warning and does **not** change the parameter.

In [ ]:
charge.value = 12
print(f"The value of the charge parameter is: {charge.value}")

For parameters which already exist (for example, a parameter created by a potential), the parameter can be fixed by setting their `fixed` attribute to `True`.

In [ ]:
sigma = Parameter(1.0, name='sigma', unit='Ang')
print('Is sigma fixed: {}'.format(sigma.fixed))
sigma.fixed = True
print('Is sigma fixed: {}'.format(sigma.fixed))

### Filtering parameters

Only parameters that are passed to `Control` (as `fit_parameters`) will be refined.  While it is simplest to pass all parameters in a `Universe` to `Control`, it is also possible to filter out a subset.  To demonstrate this, the example universe filled with SPCE water molecules is read from [Building a Universe](building-a-universe.ipynb).

The `%%capture` and `%run` commands below simply executes the [Building a Universe](building-a-universe.ipynb) notebook and captures the variables into this notebook. They are only valid if they are executed in the same folder as the [Building a Universe](building-a-universe.ipynb) notebook. Otherwise, please copy the last section of [Building a Universe](building-a-universe.ipynb) to set the same state.

In [ ]:
%%capture
# Run Building a universe notebook and hide output
%run "building-a-universe.ipynb"

There are 8 parameters in the universe:

In [ ]:
print(universe.parameters)

So while all 8 parameters can be passed when initiliasing `Control`, they can also be filtered. `Parameters` objects have a number of convenience methods to assist with this:

In [ ]:
parameters = universe.parameters
help(parameters)

For example, if only the charge parameters should be refined, `parameters` could be filtered by name:

In [ ]:
charges = parameters.filter_name('charge')
print(charges)

# charges is a Parameters object
print('\nThe class of charges is: {}'.format(type(charges)))

As each filter returns a `Parameters` object, filters can be chained together. For example, to find the potential strengths of all bonds:

In [ ]:
bond_potential_strengths = parameters.filter_name('potential_strength').filter_interaction('Bond')
print(bond_potential_strengths)

# These operations are commutative
print('\nThe order these methods are'
      ' applied does not matter: {}'.format(bond_potential_strengths
                                             == parameters.filter_interaction('Bond').filter_name('potential_strength')))

It is also possible to filter parameters based on the properties of the atoms to which they apply.  For instance, we can filter the SPCE parameters so that only parameters of interactions on H atoms are shown:

In [ ]:
H_parameters = parameters.filter_atom_attribute('name', 'H')
print(H_parameters)

Finally there is also a more flexible method (`Parameters.filter`) which can be used in conjunction with any function to filter the parameters:

In [ ]:
def is_length(parameter):
    if parameter.unit == 'Ang':
        return True
    else:
        return False
length_parameters = parameters.filter(is_length)
print(length_parameters)

# For those more familiar with Python, this can also be done using a lambda
lambda_length_parameters = parameters.filter(lambda x: x.unit == 'Ang')
print('\nThe same filter can be achieved using lambdas: {}'.format(lambda_length_parameters == length_parameters))